# Ders 12 - Ajan Not Defteri ile Sohbet Geçmişi Azaltma

Bu defter, Microsoft Agent Framework kullanarak uzun sohbetlerde bağlamın nasıl yönetileceğini gösterir. Konuşmalar büyüdükçe, token sayısı artar — sonunda modelin bağlam penceresini aşar. Buna, **bağlam özetleme deseni** ve kalıcı bellek için **ajan not defteri** ile çözüm getiriyoruz.

## Öğrenecekleriniz:
1. **Bağlam Yönetiminin Neden Önemli Olduğu**: Token sınırları ve bağlam pencerelerini anlama
2. **Bağlam Farkında Ajanlar**: Kendi sohbet bağlamını yöneten ajanlar oluşturma
3. **Bağlam Özetleme Deseni**: Konuşma geçmişini yoğunlaştırmak için araçlar kullanma
4. **Ajan Not Defteri**: Bağlam azaltma sonrası kalan kalıcı bellek

## Önkoşullar:
- Ortam değişkenleri yapılandırılmış Azure OpenAI kurulumu
- Önceki derslerden temel ajan kavramlarının anlaşılması


## Kurulum


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Neden Bağlam Yönetimi Önemlidir

Her LLM'nin sınırlı bir **bağlam penceresi** vardır — tek bir istekte işleyebileceği maksimum token sayısı. Çok turlu bir sohbet ilerledikçe:

- **Token sayısı her kullanıcı mesajı ve asistan yanıtıyla lineer olarak artar.**
- **İleti tokenları maliyeti domine eder** çünkü tüm geçmiş her turda yeniden gönderilir.
- Sonunda sohbet **bağlam penceresini aşar** ve model ya keser ya da hata verir.

### Bağlam Yönetimi için Stratejiler

| Strateji | Nasıl Çalışır | Dezavantaj |
|---|---|---|
| **Kesme** | En eski mesajları atar | Erken bağlam kaybolur |
| **Özetleme** | Daha eski mesajları özet haline getirir | Biraz detay kaybı ancak ana noktalar korunur |
| **Scratchpad / Harici Bellek** | Anahtar bilgileri sohbet dışında saklar | Araç çağrısı gerektirir, ancak her türlü azalmaya dayanır |

Bu not defterinde **özetleme** ile bir **scratchpad aracı** birleştiriyoruz, böylece ajan sohbet geçmişi yoğunlaştırılsa bile devamlılığı koruyabilir.


## Bağlam Duyarlı Bir Temsilci Oluşturma


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Uzun Bir Konuşmayı Simüle Etmek

Bağlamın nasıl biriktiğini görmek için çok aşamalı bir konuşmayı inceleyelim. Ajan, tur boyunca önemli detayları (tercihler, bütçe, seyahat tarihleri) saklamalı ve süreklilik göstermelidir.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Ajanın önceki turdan bağlamı nasıl koruduğuna dikkat edin — Japonya, suşi, tapınaklar, fotoğrafçılık, 3000$ bütçe, tek başına seyahat ve Nisan zaman dilimi hakkında bilgi sahibi. Kısa bir konuşmada bu iyi çalışır, ancak konuşma büyüdükçe tam geçmişin tekrar gönderilmesi maliyetli olur.

Bağlam birikimini görmek için konuşmaya daha fazla turla devam edelim:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Bağlam Özetleme Deseni

Konuşma ilerledikçe, biriken bağlamı sıkıştırarak kompakt bir forma dönüştürmek için bir **özetleme aracı** kullanabiliriz. Ajan, temel tercihleri kaydetmek için bu aracı çağırır, böylece eski mesajlar silinse bile, önemli bilgiler korunur.

Bu desen, daha gelişmiş geçmiş azaltma yöntemlerinin yapı taşıdır:
1. Ajan, konuşmadan temel gerçekleri belirler
2. Bunları kalıcı hale getirmek için özetleme aracını çağırır
3. Özet önemli olanları yakaladığından eski mesajlar güvenle kaldırılabilir

Aşağıda, ajanın öğrendiklerinin kompakt bir özetini kaydetmek için çağırabileceği bir `summarize_preferences` aracı tanımlıyoruz.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Özet

Bu derste Microsoft Agent Framework kullanarak uzun süreli ajan konuşmalarında bağlamı nasıl yöneteceğinizi öğrendiniz:

### Temel Kavramlar
- **Bağlam pencereleri sınırlıdır** — konuşma geçmişindeki her token maliyetlidir ve limite sayılır.
- **Özetleme araçları** ajanın biriken bağlamı kompakt özetler haline getirerek token kullanımını azaltmasını sağlar ve temel bilgileri korur.
- **Ajan karalama defterleri** konuşma kısaltmalarına rağmen kalıcı dış bellek sağlar.

### Neler Yaptınız
- Çok turlu konuşmalarda sürekliliği koruyan **bağlam farkında ajan**
- Önemli kullanıcı ayrıntılarını kompakt biçimde kaydeden bir **özetleme aracı** (`summarize_preferences`)
- Bağlam tutmayı ve değişiklik yönetimini gösteren **çok turlu bir konuşma**

### Gerçek Dünya Uygulamaları
- **Müşteri Hizmet Botları**: Uzun destek oturumları boyunca tercihleri hatırlama
- **Kişisel Asistanlar**: Bağlamı tekrar açıklamaya gerek kalmadan devam eden projeleri takip etme
- **Eğitsel Öğretmenler**: Birçok etkileşim boyunca öğrenci ilerlemesini sürdürme

### Sonraki Adımlar
- Dosya tabanlı kalıcılıkla tam bir karalama defteri aracı uygulayın
- Özetlemeden sonra otomatik geçmiş kısaltma ekleyin
- Anlamsal bellek araması için vektör veritabanları ile birleştirin
- Konuşmaları günler sonra tam bağlamla devam ettirebilen ajanlar oluşturun


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
